# Datathon Passos Mágicos — Fase 5  
## Modelo Preditivo de Risco de Defasagem

Este notebook responde à **Pergunta 9**, construindo um modelo de Machine Learning
para estimar a probabilidade de um aluno entrar em **risco de defasagem**,
utilizando indicadores educacionais, comportamentais e psicossociais.


In [32]:
import sys
import os

# adiciona a raiz do projeto ao PYTHONPATH
PROJECT_ROOT = os.path.abspath(os.path.join(".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)


In [33]:
import os
import sys
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

import joblib

from src.data_loader import load_pede_excel
from src.preprocess import unify_pede


In [34]:
DATA_PATH = "../data/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

dfs = load_pede_excel(DATA_PATH)

df = unify_pede({
    2022: dfs["PEDE2022"],
    2023: dfs["PEDE2023"],
    2024: dfs["PEDE2024"]
})

df.shape


(3030, 63)

In [35]:
def resolver_pedra(row):
    if row["ano"] == 2022:
        return row.get("pedra_22")
    elif row["ano"] == 2023:
        return row.get("pedra_23") or row.get("pedra_2023")
    elif row["ano"] == 2024:
        return row.get("pedra_2024")
    return None

df["pedra"] = df.apply(resolver_pedra, axis=1)


In [36]:
def resolver_inde(row):
    if row["ano"] == 2022:
        return row.get("inde_22")
    elif row["ano"] == 2023:
        return row.get("inde_23") or row.get("inde_2023")
    elif row["ano"] == 2024:
        return row.get("inde_2024")
    return None

df["inde"] = df.apply(resolver_inde, axis=1)


In [37]:
def resolver_pedra(row):
    if row["ano"] == 2022:
        return row.get("pedra_22")
    elif row["ano"] == 2023:
        return row.get("pedra_23") or row.get("pedra_2023")
    elif row["ano"] == 2024:
        return row.get("pedra_2024")
    return None

df["pedra"] = df.apply(resolver_pedra, axis=1)


In [38]:
df["risco_defasagem"] = (df["defasagem"] <= -1).astype(int)

df["risco_defasagem"].value_counts()


risco_defasagem
0    1944
1    1086
Name: count, dtype: int64

In [39]:
features = [
    "ida",
    "ieg",
    "ips",
    "ipp",
    "iaa",
    "ian",
    "inde"
]

# features = [
#     "ida",
#     "ieg",
#     "ips",
#     "ipp",
#     "iaa",

# ]

X = df[features]
y = df["risco_defasagem"]

X.shape, y.shape


((3030, 7), (3030,))

In [40]:
X = df[features]
y = df["risco_defasagem"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [41]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        random_state=42,
       # class_weight="balanced"
       class_weight={0:1, 1:1.3}
    ))
])


In [42]:
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("ROC AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))


ROC AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       389
           1       1.00      1.00      1.00       217

    accuracy                           1.00       606
   macro avg       1.00      1.00      1.00       606
weighted avg       1.00      1.00      1.00       606



Após ajustes de ponderação de classes, o modelo manteve ROC AUC próximo de 0,77, equilibrando melhor precisão e recall para a classe de risco.
Essa configuração reduz falsos positivos sem comprometer significativamente a identificação de alunos críticos, tornando o modelo mais adequado para uso operacional

In [43]:
import os
import joblib

os.makedirs("models", exist_ok=True)

joblib.dump(pipeline, "../models/modelo_risco_defasagem.joblib")


['../models/modelo_risco_defasagem.joblib']

In [44]:
import json
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

metricas = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred)),
    "recall": float(recall_score(y_test, y_pred)),
    "f1_score": float(f1_score(y_test, y_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_prob)),
    "features": X_train.columns.tolist()
}

with open("../models/metricas_modelo.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, ensure_ascii=False, indent=4)